In [ ]:
import numpy as np
import pandas as pd
import json

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gumbel_r

from datetime import datetime, time

In [ ]:
df=pd.read_excel('Data_Model_Pizza_Sales.xlsx')
df.head()

## customer_id

In [ ]:
df.columns,len(df)

In [ ]:
# Parámetros de la distribución de Gumbel
loc = 4000  # Parámetro de localización (media aproximada)
scale = 1000  # Parámetro de escala (desviación estándar aproximada)

# Generar valores aleatorios para customer_id
np.random.seed(42)  # Para reproducibilidad

# Generar valores según una distribución de Gumbel (Max Extreme)
gumbel_values = gumbel_r.rvs(loc=loc, scale=scale, size=len(df))

# Aplicar la probabilidad de NaN (20%)
p_nan = 0.1
customer_id = np.where(np.random.random(size=len(df)) < p_nan, np.nan, gumbel_values)

# Agregar la columna al DataFrame
df['customer_id'] = customer_id

# Verificar los valores únicos
print(len(df['customer_id'].unique()))

# Visualizar la distribución
plt.figure(figsize=(8, 6))
sns.histplot(df['customer_id'].dropna(), kde=True, color='blue', bins=30)
plt.title('Distribución de customer_id (Gumbel - Max Extreme)')
plt.xlabel('customer_id')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
int(df['customer_id'].max()),int(df['customer_id'].min())

In [ ]:
df['customer_id'] = pd.to_numeric(df['customer_id'], errors='coerce')
print(df['customer_id'].dtype) 
# Paso 2: Restar 3121 a todos los valores de 'customer_id'
df['customer_id'] = df['customer_id'] - 3389

# Paso 3: Reemplazar valores menores o iguales a 0 con NaN
df['customer_id'] = df['customer_id'].where(df['customer_id'] > 0, np.nan)
df['customer_id'] = df['customer_id'].where(df['customer_id'] <10000 , np.nan)

# Paso 4: Convertir la columna a tipo 'Int64' (que admite NaN)
df['customer_id'] = df['customer_id'].apply(lambda x: pd.NA if pd.isna(x) else int(x))

# Verificar el resultado
print(len(df['customer_id'].unique()))
print(df['customer_id'].dtype)

In [ ]:
int(df['customer_id'].max()),int(df['customer_id'].min())

In [ ]:
# Visualizar la distribución
plt.figure(figsize=(8, 6))
sns.histplot(df['customer_id'].dropna(), kde=True, color='blue', bins=30)
plt.title('Distribución de customer_id (Gumbel - Max Extreme)')
plt.xlabel('customer_id')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
print(df.isnull().sum())

## location 

In [ ]:
# Crear un DataFrame con las plazas comerciales y sus coordenadas
data = {
    "mall": [
        "Centro Comercial Santa Fe",
        "Plaza de la Constitución (Zócalo)",
        "Perisur",
        "Plaza Satélite",
        "Parque Delta",
        "Plaza Interlomas",
        "Reforma 222",
        "Plaza de la Ánimas",
        "Antara Polanco",
        "Paseo Acoxpa",
        "Mítikah",
        "Vasco de Quiroga",
        "Plaza Perisur",
        "Centro Comercial Reforma 222",
        "Plaza Satélite",
        "Plaza Las Ámericas",
        "Plaza Aragón",
        "Multiplaza Cumbres",
        "Plaza Norte",
        "El Palacio de Hierro"
    ],
    "lat": [
        19.3613, 19.4328, 19.3000, 19.4750, 19.3583, 19.4442, 19.4355, 19.3733, 19.4310, 19.3111, 
        19.3598, 19.3769, 19.2969, 19.4321, 19.4750, 19.3564, 19.5430, 19.5189, 19.6712, 19.4283
    ],
    "lon": [
        -99.2738, -99.1333, -99.1800, -99.2500, -99.1625, -99.2675, -99.1645, -99.1520, -99.1962, -99.1578,
        -99.1686, -99.2463, -99.1792, -99.1645, -99.2500, -99.2214, -99.0336, -99.1836, -99.1563, -99.1833
    ]
}

# Crear el DataFrame
df_plazas_comerciales = pd.DataFrame(data)


In [ ]:
np.random.seed(0)  # Para reproducibilidad

probabilidades = np.random.dirichlet(np.ones(20), size=1)[0]
plazas_asignadas = np.random.choice(df_plazas_comerciales['mall'], size=len(df),p=probabilidades)
df['mall'] = plazas_asignadas

# Hacer un merge para agregar latitud y longitud
df = df.merge(df_plazas_comerciales, on='mall', how='left')

# Verificar el resultado
df.head()

In [ ]:
probabilidades

In [ ]:
for i in df['order_id'].unique():
    cliente=df[df['order_id']==i]['customer_id'].unique()[0]
    if pd.notnull(cliente):
        df.loc[df['order_id'] == i, 'customer_id'] = cliente


In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df['customer_id'].dropna(), kde=True, color='blue', bins=30)
plt.title('Distribución de customer_id (Gumbel - Max Extreme)')
plt.xlabel('customer_id')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
for i in df['customer_id'].unique():
    subset = df[df['customer_id'] == i]
    
    if not subset.empty:
        plaza = subset[['mall', 'lat', 'lon']].iloc[0]
        
        df.loc[df['customer_id'] == i, ['mall', 'lat', 'lon']] = plaza.values


In [ ]:
df.head()

In [ ]:
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df['mall'], kde=True, color='blue', bins=30)
plt.title('Distribución de plazas')
plt.xlabel('customer_id')
plt.xticks(rotation=90)
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Supongamos que df es tu DataFrame y tiene las columnas 'mall', 'pizza_id' y 'order_id'

# 1. Identificar la pizza más popular en cada sucursal
pizza_maxima_por_mall = (
    df.groupby(['mall', 'pizza_id'])['order_id']
    .count()
    .reset_index(name='cantidad')
    .sort_values(by=['mall', 'cantidad'], ascending=[True, False])
    .drop_duplicates(subset=['mall'], keep='first')  # Mantener solo la pizza más popular por mall
)

# 2. Crear un diccionario con la pizza máxima por mall
pizza_maxima_dict = dict(zip(pizza_maxima_por_mall['mall'], pizza_maxima_por_mall['pizza_id']))
pizza_maxima_dict

In [ ]:
np.random.seed(42)  # Para reproducibilidad


# Supongamos que df es tu DataFrame y tiene las columnas 'mall', 'pizza_id' y 'order_id'

# 1. Identificar la pizza más popular en cada sucursal
pizza_maxima_por_mall = (
    df.groupby(['mall', 'pizza_id'])['order_id']
    .count()
    .reset_index(name='cantidad')
    .sort_values(by=['mall', 'cantidad'], ascending=[True, False])
    .drop_duplicates(subset=['mall'], keep='first')  # Mantener solo la pizza más popular por mall
)

# 2. Crear un diccionario con la pizza máxima por mall
pizza_maxima_dict = dict(zip(pizza_maxima_por_mall['mall'], pizza_maxima_por_mall['pizza_id']))

# 3. Función para cambiar el 10% de las pizzas al pizza_id máximo de cada mall
def cambiar_a_pizza_maxima(group):
    mall = group.name  # Obtener el nombre del grupo (mall)
    pizza_maxima = pizza_maxima_dict[mall]  # Obtener la pizza máxima de este mall
    
    # Seleccionar aleatoriamente el 10% de las filas en este grupo
    n_cambios = int(len(group) * 0.10)  # 10% de las pizzas
    if n_cambios > 0:  # Asegurarse de que haya al menos una pizza para cambiar
        indices_a_cambiar = np.random.choice(group.index, size=n_cambios, replace=False)
        
        # Cambiar el pizza_id de las filas seleccionadas al pizza_id máximo
        group.loc[indices_a_cambiar, 'pizza_id'] = pizza_maxima
    
    return group

# 4. Aplicar la función a cada grupo (mall)
df = df.groupby('mall', group_keys=False).apply(cambiar_a_pizza_maxima)

# 5. Verificar el resultado
print(df[['mall', 'pizza_id']])

## Tiempo elaboración 

In [ ]:
pizza_ids_unicos=df['pizza_id'].unique()
np.random.seed(42)  # Para reproducibilidad

media = 25  
desviacion_estandar = 2  

tiempos_elaboracion = {
    pizza_id: np.random.normal(media, desviacion_estandar)
    for pizza_id in pizza_ids_unicos
}


tiempos_elaboracion = {
    pizza_id: max(20, min(30, tiempo))  
    for pizza_id, tiempo in tiempos_elaboracion.items()
}

df['t_prep'] = df['pizza_id'].map(tiempos_elaboracion)

In [ ]:
df['t_prep'].unique()

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df['t_prep'], kde=True, color='blue', bins=30)
plt.title('Distribución de tiempo de pizzas')
plt.xlabel('customer_id')
plt.xticks(rotation=90)
plt.ylabel('Frecuencia')
plt.show()

## Mejorar distribución

In [ ]:
df.columns

In [ ]:
df['pizza_size'].unique()

In [ ]:
# Tamaño de pizza 
np.random.seed(42)  # Para reproducibilidad

es_pizza_S = df['pizza_size'] == 'S'
es_pizza_M = df['pizza_size'] == 'M'
es_pizza_L = df['pizza_size'] == 'L'
es_pizza_XL = df['pizza_size'] == 'XL'
es_pizza_XXL = df['pizza_size'] == 'XXL'

random_S = np.random.uniform(-2, 0, size=es_pizza_S.sum())
random_M = np.random.uniform(-1, 0, size=es_pizza_M.sum())
random_L = np.random.uniform(0, 1, size=es_pizza_L.sum())
random_XL = np.random.uniform(0, 2, size=es_pizza_XL.sum())
random_XXL = np.random.uniform(1, 3, size=es_pizza_XXL.sum())


df.loc[es_pizza_S, 't_prep'] += random_S
df.loc[es_pizza_M, 't_prep'] += random_M
df.loc[es_pizza_L, 't_prep'] += random_L
df.loc[es_pizza_XL, 't_prep'] += random_XL
df.loc[es_pizza_XXL, 't_prep'] += random_XXL

# Verificar el DataFrame resultante
print(df[['pizza_size', 't_prep']])

In [ ]:
# Supongamos que df es tu DataFrame y tiene las columnas 'order_time' y 't_prep'

# 1. Convertir 'order_time' a datetime si no está en ese formato
#df['order_time'] = pd.to_datetime(df['order_time'])

# 2. Definir una hora de referencia para calcular cuán tarde es el pedido
# Por ejemplo, supongamos que la hora de referencia es las 18:00 (6 PM)
hora_referencia = time(18, 0)  # 6 PM

# 3. Calcular cuántas horas después de la hora de referencia se hizo el pedido
df['horas_despues'] = df['order_time'].apply(
    lambda x: max(0, (x.hour - hora_referencia.hour) + (x.minute / 60)))
# Nota: x.hour y x.minute ya son propiedades de datetime, no necesitas .time()

# 4. Aumentar el tiempo de preparación en función de las horas después de la hora de referencia
# Por ejemplo, aumentar 0.5 minutos por cada hora después de la hora de referencia
factor_aumento = 0.5  # Aumento en minutos por cada hora
df['t_prep'] += df['horas_despues'] * factor_aumento

# 5. Verificar el DataFrame resultante
df[['order_time', 'horas_despues', 't_prep']]

In [ ]:
df.drop(columns=['horas_despues'], inplace=True)

df.columns

In [ ]:
import pandas as pd

# Supongamos que df es tu DataFrame y tiene las columnas 'mall', 'pizza_id' y 'order_id'

# 1. Agrupar por 'mall' y 'pizza_id', y contar las ocurrencias
pizzas_favoritas = (
    df.groupby(['mall', 'pizza_id'])['order_id']  # Agrupar por 'mall' y 'pizza_id'
    .count()  # Contar las ocurrencias
    .reset_index(name='cantidad')  # Convertir a DataFrame y renombrar la columna de conteo
    .sort_values(by=['mall', 'cantidad'], ascending=[True, False])  # Ordenar por 'mall' y 'cantidad' (descendente)
)

# 2. Obtener las 3 pizzas favoritas de cada 'mall'
top_3_pizzas_por_mall = (
    pizzas_favoritas.groupby('mall')  # Agrupar por 'mall'
    .head(3)  # Tomar las 3 primeras filas de cada grupo
    .reset_index(drop=True)  # Reiniciar el índice
)

# 3. Verificar el resultado
print(top_3_pizzas_por_mall)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Supongamos que top_3_pizzas_por_mall es el DataFrame con las 3 pizzas favoritas por mall

# 1. Configurar el estilo de la gráfica
sns.set(style="whitegrid")

# 2. Crear la gráfica de barras
plt.figure(figsize=(10, 6))  # Tamaño de la gráfica
sns.barplot(
    x='mall',  # Eje X: mall
    y='cantidad',  # Eje Y: cantidad
    hue='pizza_id',  # Diferenciar por pizza_id
    data=top_3_pizzas_por_mall,  # Datos
    palette='viridis'  # Paleta de colores
)

# 3. Personalizar la gráfica
plt.title('Top 3 pizzas favoritas por mall', fontsize=16)  # Título
plt.xlabel('Mall', fontsize=14)  # Etiqueta del eje X
plt.ylabel('Cantidad de pedidos', fontsize=14)  # Etiqueta del eje Y
plt.legend(title='Pizza ID', bbox_to_anchor=(1.05, 1), loc='upper left')  # Leyenda fuera del gráfico
plt.tight_layout()  # Ajustar el layout
plt.xticks(rotation=90)
# 4. Mostrar la gráfica
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df['t_prep'], kde=True, color='blue', bins=30)
plt.title('Distribución de tiempo de pizzas')
plt.xlabel('t_prep')
plt.xticks(rotation=90)
plt.ylabel('Frecuencia')
plt.show()

## Pasar a json

In [ ]:
df['pizza_ingredients'][0]

In [ ]:
df.columns

In [ ]:
from datetime import time
def remove_nan(obj):
    if isinstance(obj, dict):
        return {k: remove_nan(v) for k, v in obj.items() if v is not None and not (isinstance(v, float) and np.isnan(v))}
    elif isinstance(obj, list):
        return [remove_nan(item) for item in obj if item is not None and not (isinstance(item, float) and np.isnan(item))]
    elif isinstance(obj, pd.Timestamp):  # Convertir Timestamp a string
        return obj.strftime('%Y-%m-%d')  # O el formato que prefieras
    elif isinstance(obj, time):  # Convertir time a string
        return obj.strftime('%H:%M:%S')  # O el formato que prefieras
    else:
        return obj

dict_data = df.to_dict(orient='records')

cleaned_data = [remove_nan(record) for record in dict_data]

json_data = json.dumps(cleaned_data, indent=4)

print(json_data)

In [ ]:
df['pizza_ingredients'] = df['pizza_ingredients'].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
gramos_por_ingrediente={}
df['pizza_ingredients']

In [ ]:
ingredientes_desanidados = df.explode('pizza_ingredients')

# 2. Obtener los ingredientes únicos
ingredientes_unicos = ingredientes_desanidados['pizza_ingredients'].dropna().unique()

# 3. Verificar los ingredientes únicos
print("Ingredientes únicos:", ingredientes_unicos)

In [ ]:
df['pizza_ingredients']

In [ ]:
# Definir tamaños de pizza
tamanos_pizza = ['S', 'M', 'L', 'XL', 'XXL']

# Definir ingredientes
ingredientes = [
    'Sliced Ham', 'Pineapple', 'Mozzarella Cheese', 'Pepperoni', 'Mushrooms',
    'Red Onions', 'Red Peppers', 'Bacon', 'Provolone Cheese', 'Smoked Gouda Cheese',
    'Romano Cheese', 'Blue Cheese', 'Garlic', 'Calabrese Salami', 'Capocollo',
    'Tomatoes', 'Green Olives', 'Jalapeno Peppers', 'Cilantro', 'Corn',
    'Chipotle Sauce', 'Chicken', 'Thai Sweet Chilli Sauce', 'Prosciutto di San Daniele',
    'Arugula', 'Barbecued Chicken', 'Green Peppers', 'Barbecue Sauce', 'Kalamata Olives',
    'Feta Cheese', 'Beef Chuck Roast', 'Spinach', 'Artichokes', 'Asiago Cheese',
    'Goat Cheese', 'Oregano', 'Peperoncini verdi', 'Sun-dried Tomatoes', 'Pesto Sauce',
    'Zucchini', 'Artichoke', 'Fontina Cheese', 'Gouda Cheese', 'Italian Sausage',
    'Chorizo Sausage', 'Soppressata Salami', 'Ricotta Cheese', 'Gorgonzola Piccante Cheese',
    'Parmigiano Reggiano Cheese', 'Anchovies', 'Nduja Salami', 'Pancetta', 'Friggitello Peppers',
    'Eggplant', 'Plum Tomatoes', 'Genoa Salami', 'Coarse Sicilian Salami', 'Luganega Sausage',
    'Onions', 'Alfredo Sauce', 'Brie Carre Cheese', 'Prosciutto', 'Caramelized Onions',
    'Pears', 'Thyme'
]

# Crear el diccionario gramos_por_ingrediente
gramos_por_ingrediente = {}

# Asignar gramos según el tamaño de la pizza
for ingrediente in ingredientes:
    gramos_por_ingrediente[ingrediente] = {
        'S': 10,   # Pequeña cantidad para tamaño S
        'M': 20,   # Cantidad moderada para tamaño M
        'L': 30,   # Cantidad estándar para tamaño L
        'XL': 40,  # Cantidad generosa para tamaño XL
        'XXL': 50  # Cantidad abundante para tamaño XXL
    }

# Ajustar gramos para ingredientes específicos
# Por ejemplo, los quesos y carnes suelen tener más gramos
quesos = ['Mozzarella Cheese', 'Provolone Cheese', 'Smoked Gouda Cheese', 'Romano Cheese',
          'Blue Cheese', 'Feta Cheese', 'Asiago Cheese', 'Goat Cheese', 'Fontina Cheese',
          'Gouda Cheese', 'Ricotta Cheese', 'Gorgonzola Piccante Cheese', 'Parmigiano Reggiano Cheese',
          'Brie Carre Cheese']
carnes = ['Sliced Ham', 'Pepperoni', 'Bacon', 'Calabrese Salami', 'Capocollo', 'Chicken',
          'Prosciutto di San Daniele', 'Barbecued Chicken', 'Beef Chuck Roast', 'Italian Sausage',
          'Chorizo Sausage', 'Soppressata Salami', 'Nduja Salami', 'Pancetta', 'Genoa Salami',
          'Coarse Sicilian Salami', 'Luganega Sausage', 'Prosciutto']

for ingrediente in carnes:
    gramos_por_ingrediente[ingrediente] = {
        'S': 15,   # Más gramos para quesos y carnes en tamaño S
        'M': 30,   # Más gramos para quesos y carnes en tamaño M
        'L': 50,   # Más gramos para quesos y carnes en tamaño L
        'XL': 70,  # Más gramos para quesos y carnes en tamaño XL
        'XXL': 80 # Más gramos para quesos y carnes en tamaño XXL
    }


for ingrediente in quesos:
    gramos_por_ingrediente[ingrediente] = {
        'S': 20,   # Más gramos para quesos y carnes en tamaño S
        'M': 40,   # Más gramos para quesos y carnes en tamaño M
        'L': 60,   # Más gramos para quesos y carnes en tamaño L
        'XL': 80,  # Más gramos para quesos y carnes en tamaño XL
        'XXL': 100 # Más gramos para quesos y carnes en tamaño XXL
    }

# Verificar el diccionario
print("Gramos por ingrediente:")
for ingrediente, gramos in gramos_por_ingrediente.items():
    print(f"{ingrediente}: {gramos}")

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import time

# Función para eliminar NaN y manejar tipos especiales
def remove_nan(obj):
    if isinstance(obj, dict):
        return {k: remove_nan(v) for k, v in obj.items() if v is not None and not (isinstance(v, float) and np.isnan(v))}
    elif isinstance(obj, list):
        return [remove_nan(item) for item in obj if item is not None and not (isinstance(item, float) and np.isnan(item))]
    elif isinstance(obj, pd.Timestamp):  # Convertir Timestamp a string
        return obj.strftime('%Y-%m-%d')  # O el formato que prefieras
    elif isinstance(obj, time):  # Convertir time a string
        return obj.strftime('%H:%M:%S')  # O el formato que prefieras
    else:
        return obj


# Convertir el DataFrame a un diccionario
dict_data = df.to_dict(orient='records')

# Modificar la estructura para que 'ingredientes' sea un subdiccionario
for record in dict_data:
    if 'pizza_ingredients' in record and isinstance(record['pizza_ingredients'], list):
        # Obtener el tamaño y tipo de pizza (asumiendo que están en el registro)
        tamaño_pizza = record.get('tamaño_pizza', 'M')  # Por defecto 'M' si no está especificado
        tipo_pizza = record.get('tipo_pizza', 'Regular')  # Por defecto 'Regular' si no está especificado
        
        # Convertir la lista de ingredientes en un subdiccionario con gramos
        record['pizza_ingredients'] = {
            ingrediente: f"{gramos_por_ingrediente.get(ingrediente, {}).get(tamaño_pizza, 0)}g"
            for ingrediente in record['pizza_ingredients']
        }

# Limpiar los datos (eliminar NaN y manejar tipos especiales)
cleaned_data = [remove_nan(record) for record in dict_data]

# Convertir a JSON
json_data = json.dumps(cleaned_data, indent=4)

# Imprimir el JSON resultante
print(json_data)

In [ ]:
with open('pizza.json', 'w') as file:
    file.write(json_data)

print("El archivo JSON ha sido guardado como 'pizza.json'.")

In [ ]:
#df.to_csv('pizza_tuneada.csv', index=False, encoding='utf-8')

In [ ]:
df.columns

In [ ]:
df[['order_details_id', 'order_id', 'pizza_id', 'quantity', 'order_date',
       'order_time', 'unit_price', 'total_price', 'pizza_size',
       'pizza_category', 'pizza_name', 'customer_id',
       'mall', 'lat', 'lon', 't_prep']].duplicated().sum()

In [ ]:
json_data